# Phase 2: Snapshot Triangle Build

Just the core mechanism: what it actually takes to turn transactions into a snapshot triangle. 

## Constants

In [1]:
import math
import pandas as pd

pd.options.display.float_format = '{:,.2f}'.format

CUTOFF = 40

# Only two grid spacings are sanctioned -- matching how insurers actually
# review claims (quarterly or annually), never an arbitrary interval. Periods
# are already quarters (see CLAUDE.md), so "quarterly" is every native period
# and "yearly" is every 4th (4 quarters = 12 months). build_snapshot_triangle()
# and claim_as_triangle() both take granularity="quarterly"/"yearly" directly --
# looking it up in this dict is what rejects anything else, no separate check
# needed.
GRANULARITY_TO_STEP = {"quarterly": 1, "yearly": 4}

## Time and grid helpers

In [2]:
def ceil_period(t):
    """Round a continuous time value up to its period bucket (period=quarter)."""
    return math.ceil(t) if t != int(t) else max(1, int(t))


def load_visible(df, cutoff=CUTOFF):
    """Everything the build is allowed to see. Enforced by dropping the rows at
    load time, not by discipline downstream."""
    visible = df[df["payment_period"] <= cutoff].copy()
    # real payments are whole cents; round here once so every downstream sum is
    # cent-exact and stays consistent regardless of summation order
    visible["payment_size"] = visible["payment_size"].round(2)
    return visible


def snapshot_grid(cutoff=CUTOFF, step=GRANULARITY_TO_STEP["quarterly"]):
    """The shared grid every claim's snapshots (and observation ages) are drawn
    from: step, 2*step, ... <= cutoff. One grid for the whole portfolio, not
    per-claim offsets, so per-cluster aggregate triangles built later don't
    have misaligned rows.

    Example:
        >>> snapshot_grid(cutoff=40, step=4)
        [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]

        >>> snapshot_grid(cutoff=40, step=1)
        [1, 2, 3, 4, ..., 40]   # every single period

    A claim doesn't get its own personal grid -- it just joins this one
    shared list at whichever stop comes first at or after its notification
    period, like catching the next scheduled bus rather than calling one
    just for you. See build_snapshot_triangle()'s `entry` line for exactly
    where that "which stop do I join at" decision happens.
    """
    last = cutoff - (cutoff % step)
    return list(range(step, last + 1, step))

## The build

In [ ]:
def build_snapshot_triangle(df, cutoff=CUTOFF, granularity="quarterly"):
    step = GRANULARITY_TO_STEP[granularity]  # bad input just raises KeyError -- no separate check needed
    visible = load_visible(df, cutoff)
    grid = snapshot_grid(cutoff, step)

    # claim identity (occurrence, notidel, setldel) comes from the FULL data, not
    # `visible` -- a claim notified before the cutoff but with zero payments yet
    # (nothing due until after the cutoff) has zero rows in `visible` and would
    # silently vanish otherwise, even though it's a valid "known, nothing paid
    # yet" claim that belongs in the triangle
    claims = df.drop_duplicates("claim_no").copy()
    claims["notification_period"] = (claims["occurrence_time"] + claims["notidel"]).apply(ceil_period)
    claims["settlement_period"] = (
        claims["occurrence_time"] + claims["notidel"] + claims["setldel"]
    ).apply(ceil_period)

    n_before = len(claims)
    # notified AT the wall (not just after it) is also excluded: with nothing but the
    # wall itself left, there's no room for even one observed row to show development.
    # (The ~2 such claims that also happen to settle exactly at the wall would still
    # get a single settled marker row otherwise -- not worth keeping for that alone,
    # since a settled-only row has no future_paid target and isn't usable for training.)
    claims = claims[claims["notification_period"] < grid[-1]]
    excluded = n_before - len(claims)
    print(f"portfolio: {n_before} claims")
    print(f"excluded (notified at or after the last grid point, {grid[-1]}): "
          f"{excluded} ({excluded/n_before*100:.1f}%) -- no room left to show any development")

    rows = []
    for claim in claims.itertuples():
        payments = visible[visible["claim_no"] == claim.claim_no][["payment_period", "payment_size"]].values

        def cum_paid(p, _payments=payments):
            return _payments[_payments[:, 0] <= p][:, 1].sum()

        # first grid point at or after notification -- this claim's entry snapshot
        entry = next((grid_point for grid_point in grid if grid_point >= claim.notification_period), None)
        if entry is None:
            continue

        # exit is the claim's TRUE settlement_period, where that's at or before the
        # cutoff -- NOT "outstanding <= 0 derived from cum_paid(cutoff)". The latter
        # reads as 0 for any claim with nothing paid *yet* inside the visible window,
        # which would wrongly flag every such immature claim as settled.
        exit_period = claim.settlement_period if claim.settlement_period <= cutoff else None

        for snapshot in grid:
            if snapshot < entry:
                continue

            paid_to_date = cum_paid(snapshot)

            if exit_period is not None and snapshot >= exit_period:
                rows.append(dict(
                    claim_no=claim.claim_no, snapshot_period=snapshot, observation_age=None,
                    observation_period=None, row_status="settled", future_paid=None,
                    paid_to_date=round(paid_to_date, 2),
                ))
                break  # one exit row, then stop -- no further snapshots for this claim

            for observation_period in grid:
                if observation_period <= snapshot:
                    continue
                if observation_period > cutoff:
                    break
                k = observation_period - snapshot
                future_paid = round(cum_paid(observation_period) - paid_to_date, 2)
                rows.append(dict(
                    claim_no=claim.claim_no, snapshot_period=snapshot, observation_age=k,
                    observation_period=observation_period, row_status="observed", future_paid=future_paid,
                    paid_to_date=round(paid_to_date, 2),
                    is_settled_at_obs=(observation_period >= claim.settlement_period),
                ))

    result = pd.DataFrame(rows)
    n_observed = (result["row_status"] == "observed").sum()
    n_settled = (result["row_status"] == "settled").sum()
    print(f"snapshot triangle: {len(result)} rows ({n_observed} observed, {n_settled} settled-exit), "
          f"{result['claim_no'].nunique()} claims represented")
    return result

## Display helper: render one claim back into triangle shape

In [4]:
# GRANULARITY_TO_STEP is defined once, up in Constants, and shared by both
# build_snapshot_triangle()'s safeguard and this display function.

def claim_as_triangle(snapshot_triangle, claim_no, value_col="future_paid",
                       truncate_at_settlement=True, granularity="quarterly"):
    """Render one claim's rows back into triangle shape (snapshot rows x age columns).

    granularity: "quarterly" (every period -- the native grid, and periods are
    already quarters), "yearly" (every 4th period = 12 months), or pass an int
    step directly. FILTERS an already-built triangle down to a coarser display
    cadence -- doesn't rebuild anything.

    Display-only. truncate_at_settlement=True (default) hides ages beyond the
    claim's own settlement. Never affects what's written; pass False to see the
    full, untruncated stored rows.

    The claim's settled/exit row -- and any other on-grid snapshot with zero
    surviving age columns at this cadence -- is always included as a row
    (every age column blank), never silently dropped.
    """
    step = GRANULARITY_TO_STEP.get(granularity, granularity) if isinstance(granularity, str) else granularity

    all_rows = snapshot_triangle[snapshot_triangle.claim_no == claim_no]
    observed_native = all_rows[all_rows.row_status == "observed"]
    settled_rows = all_rows[all_rows.row_status == "settled"]

    on_grid_snapshots = observed_native[observed_native.snapshot_period % step == 0]
    claim_rows = observed_native[
        (observed_native.snapshot_period % step == 0) & (observed_native.observation_age % step == 0)
    ]

    if truncate_at_settlement:
        first_settled_age = (
            claim_rows[claim_rows["is_settled_at_obs"].astype(bool)]
            .groupby("snapshot_period")["observation_age"].min()
        )
        cutoff_age = claim_rows["snapshot_period"].map(first_settled_age)
        claim_rows = claim_rows[cutoff_age.isna() | (claim_rows["observation_age"] <= cutoff_age)]

    paid_so_far = pd.concat([
        on_grid_snapshots.groupby("snapshot_period")["paid_to_date"].first(),
        settled_rows.set_index("snapshot_period")["paid_to_date"],
    ]).rename("paid_to_date")

    pivot = claim_rows.pivot(index="snapshot_period", columns="observation_age", values=value_col)
    if pivot.empty:
        result = paid_so_far.to_frame()
    else:
        result = pd.concat([paid_so_far, pivot], axis=1)
    return result.sort_index()

## Run it on the real data

In [5]:
df = pd.read_csv('../data/synthetic_transactions_with_covariates.csv')
snapshot_triangle = build_snapshot_triangle(df)

portfolio: 3624 claims
excluded (notified at or after the last grid point, 40): 301 (8.3%) -- no room left to show any development
snapshot triangle: 382485 rows (379840 observed, 2645 settled-exit), 3323 claims represented


In [6]:
snapshot_triangle.head(10)

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
0,1,2,1.00,3.00,observed,0.00,0.00,False
1,1,2,2.00,4.00,observed,0.00,0.00,False
2,1,2,3.00,5.00,observed,0.00,0.00,False
3,1,2,4.00,6.00,observed,0.00,0.00,False
4,1,2,5.00,7.00,observed,0.00,0.00,False
5,1,2,6.00,8.00,observed,0.00,0.00,False
6,1,2,7.00,9.00,observed,"20,536.32",0.00,False
7,1,2,8.00,10.00,observed,"20,536.32",0.00,False
8,1,2,9.00,11.00,observed,"38,078.57",0.00,False
9,1,2,10.00,12.00,observed,"38,078.57",0.00,False


## Viewing a claim as a triangle

In [11]:
df.head(5)

,claim_no,pmt_no,occurrence_period,occurrence_time,claim_size,notidel,setldel,payment_time,payment_period,payment_size,payment_inflated,payment_delay,legal_representation,injury_severity,age_of_claimant
0,1,1,1,0.62,"294,223.32",0.93,16.29,8.48,9,"20,536.32","21,416.34",6.92,Y,1,50-65
1,1,2,1,0.62,"294,223.32",0.93,16.29,10.29,11,"17,542.25","18,459.30",1.82,Y,1,50-65
2,1,3,1,0.62,"294,223.32",0.93,16.29,16.89,17,"236,377.45","256,990.79",6.60,Y,1,50-65
3,1,4,1,0.62,"294,223.32",0.93,16.29,17.84,18,"19,767.30","21,592.92",0.95,Y,1,50-65
4,2,1,1,0.12,"236,808.92",0.59,28.06,4.84,5,"7,270.62","7,446.81",4.12,Y,2,0-15


In [7]:
claim_as_triangle(snapshot_triangle, claim_no=1, granularity='quarterly')

,paid_to_date,1.00,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00,10.00,11.00,12.00,13.00,14.00,15.00,16.00
snapshot_period,,,,,,,,,,,,,,,,,
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32"
3,0.00,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN
4,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN
5,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN
6,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN
7,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN,NaN
8,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN,NaN,NaN
9,"20,536.32",0.00,"17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","253,919.70","273,687.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,"20,536.32","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","253,919.70","273,687.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
claim_as_triangle(snapshot_triangle, claim_no=1, granularity='yearly')

,paid_to_date,4.00,8.00,12.00,16.00
snapshot_period,,,,,
4,0.00,0.00,"38,078.57","38,078.57","294,223.32"
8,0.00,"38,078.57","38,078.57","294,223.32",NaN
12,"38,078.57",0.00,"256,144.75",NaN,NaN
16,"38,078.57","256,144.75",NaN,NaN,NaN
18,"294,223.32",NaN,NaN,NaN,NaN


In [9]:
# the exit row -- claim #1 settles at true period 19, and 19 is itself a grid point at quarterly granularity
snapshot_triangle[(snapshot_triangle.claim_no == 1) & (snapshot_triangle.row_status == 'settled')]

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
488,1,18,NaN,NaN,settled,NaN,"294,223.32",NaN


## Row status breakdown across the whole portfolio

In [10]:
snapshot_triangle['row_status'].value_counts()

row_status
observed    379840
settled       2645
Name: count, dtype: int64